# Titanic Survival Classification

A compact end-to-end ML portfolio project. Download Kaggle competition data, train a reproducible pipeline, evaluate it, and save the result to Google Drive.

## 1. Install dependencies

This notebook is intended for Google Colab.

In [ ]:
!pip -q install kaggle joblib seaborn scikit-learn


## 2. Authenticate and download the Kaggle dataset

Create an API token in Kaggle Settings, then upload the downloaded `kaggle.json` file below. **Never commit this file to GitHub.**

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload kaggle.json

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!mkdir -p data
!kaggle competitions download -c titanic -p data
!unzip -o data/titanic.zip -d data


## 3. Load and inspect the training data

In [ ]:
import pandas as pd

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
display(train.head())
print(f'Training rows: {len(train):,} | Test rows: {len(test):,}')
train.info()


## 4. Build a reproducible preprocessing and model pipeline

The pipeline imputes missing values, one-hot encodes categorical features, and trains a Logistic Regression baseline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'
numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
categorical_features = ['Sex', 'Embarked']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42)),
])

X_train, X_valid, y_train, y_valid = train_test_split(
    train[features], train[target], test_size=0.2, stratify=train[target], random_state=42
)
model.fit(X_train, y_train)


## 5. Evaluate the baseline

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

predictions = model.predict(X_valid)
print(f'Validation accuracy: {accuracy_score(y_valid, predictions):.3f}')
print(classification_report(y_valid, predictions))
ConfusionMatrixDisplay.from_predictions(y_valid, predictions, cmap='Blues')
plt.title('Titanic Survival: Validation Confusion Matrix')
plt.show()


## 6. Save the trained pipeline to Google Drive

The saved pipeline includes both preprocessing and the classifier, so it is ready for future predictions.

In [ ]:
from google.colab import drive
import joblib
from pathlib import Path

drive.mount('/content/drive')
model_dir = Path('/content/drive/MyDrive/ml-portfolio')
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / 'titanic_survival_pipeline.joblib'
joblib.dump(model, model_path)
print(f'Model saved to: {model_path}')


## 7. Optional: create Kaggle submission predictions

In [ ]:
submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': model.predict(test[features]),
})
submission.to_csv('titanic_submission.csv', index=False)
submission.head()
